# 2.3. Đánh giá hiệu năng phân tích cấu trúc trên Google Colab CPU-only

## 1. Kết quả đo 5 phiên độc lập và đối chiếu dung lượng lưu trữ

Cả 5 phiên phân tích thành viên (mỗi phiên xử lý mới 5.000 bài báo, `publications_reused = 0`, `sessions_overlap = False`) đều thực thi trên máy ảo Google Colab Standard CPU-only (2 vCPU Intel x86_64, tổng RAM 12.975,52 MiB, Python 3.13.18) và được kiểm tra tự động qua `src/crawler/benchmark_sessions.py`.

In [ ]:
import html
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

# Bảng màu học thuật tối giản: 1 tông xanh đậm chủ đạo kết hợp thang xám điềm đạm
BOOK_PALETTE = {
    "navy": "#1B365D",
    "steel": "#3B6E8C",
    "slate": "#64748B",
    "muted": "#94A3B8",
    "light": "#CBD5E1",
    "accent": "#2C5282",
    "grid": "#CBD5E1",
}
MEMBER_STYLES = {
    "DAT": ("#1B365D", "-"),
    "TUAN": ("#3B6E8C", "--"),
    "TAN": ("#4A5568", "-."),
    "TRI": ("#64748B", ":"),
    "TRIET": ("#94A3B8", "-"),
}

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans", "DejaVu Sans", "Arial"],
    "figure.figsize": (9.6, 4.0),
    "figure.dpi": 170,
    "savefig.dpi": 170,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 10.5,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#1B365D",
    "axes.labelsize": 9.2,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.6,
    "ytick.labelsize": 8.6,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": ":",
    "grid.alpha": 0.50,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "-"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    **_unused,
):
    # Bảng học thuật kiểu booktabs tối giản: không thanh màu nền, không nút sắp xếp, an toàn HTML 100%.
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    num_cols: set[str] = set()
    for c in out.columns:
        if c in int_set or c in pct_set or pd.api.types.is_numeric_dtype(out[c]):
            num_cols.add(c)

    parts = ['<div class="ah-table-scroll"><table class="ah-clean-table">']
    if caption:
        parts.append(f"<caption>{html.escape(str(caption))}</caption>")

    parts.append("<thead><tr>")
    for col in out.columns:
        cls = ' class="num-col"' if col in num_cols else ""
        parts.append(f"<th{cls}>{html.escape(str(col))}</th>")
    parts.append("</tr></thead><tbody>")

    for _, row in out.iterrows():
        parts.append("<tr>")
        for col in out.columns:
            val = row[col]
            if col in custom_fmt:
                txt = custom_fmt[col](val)
            elif isinstance(val, (list, tuple)):
                txt = ", ".join(str(x) for x in val) if val else "-"
            elif val is None or (not isinstance(val, (str, bytes)) and pd.isna(val)):
                txt = "-"
            elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
                txt = fmt_int(val)
            elif col in pct_set:
                txt = fmt_pct(val, precision, pct_scale100)
            elif pd.api.types.is_float_dtype(out[col]):
                txt = fmt_num(val, precision)
            else:
                txt = str(val)
            cls = ' class="num-col"' if col in num_cols else ""
            esc = html.escape(str(txt))
            if not cls and str(txt).startswith(("http://", "https://", "10.")):
                esc = f'<span class="ah-breakable">{esc}</span>'
            parts.append(f"<td{cls}>{esc}</td>")
        parts.append("</tr>")

    parts.append("</tbody></table></div>")
    return HTML("".join(parts))


def draw_pipeline_diagram(title: str, stages: list[tuple[str, str]]) -> None:
    # Sơ đồ khối quy trình tối giản chuẩn báo cáo khoa học.
    fig, ax = plt.subplots(figsize=(10.0, 2.05))
    ax.set_xlim(0, 10.0)
    ax.set_ylim(0, 2.05)
    ax.axis("off")
    n = len(stages)
    bw = 2.22
    gap = (9.70 - n * bw) / max(n - 1, 1)
    for idx, (st_title, st_desc) in enumerate(stages):
        x0 = 0.15 + idx * (bw + gap)
        ax.add_patch(Rectangle((x0, 0.18), bw, 1.56, facecolor="#FFFFFF", edgecolor="#CBD5E1", linewidth=1.0))
        ax.add_patch(Rectangle((x0, 1.32), bw, 0.42, facecolor="#EDF2F7", edgecolor="#CBD5E1", linewidth=0.8))
        ax.add_patch(Rectangle((x0, 1.32), 0.06, 0.42, facecolor="#1B365D", edgecolor="none"))
        ax.text(x0 + bw / 2 + 0.02, 1.53, st_title, ha="center", va="center", color="#1B365D", fontsize=8.8, fontweight="bold")
        ax.text(x0 + bw / 2, 0.74, st_desc, ha="center", va="center", color="#334155", fontsize=8.0, linespacing=1.35)
        if idx < n - 1:
            ax.add_patch(FancyArrowPatch(
                (x0 + bw + 0.04, 0.96),
                (x0 + bw + gap - 0.04, 0.96),
                arrowstyle="-|>,head_length=5,head_width=3.2",
                color="#64748B",
                lw=1.1,
            ))
    ax.set_title(title, pad=4, fontsize=10.0, fontweight="bold", color="#1B365D")
    plt.tight_layout()
    plt.show()

In [ ]:
tot = bench["totals"]
ep = m2["tex_elapsed_percentiles"]
df_b8 = pd.DataFrame([
    ("Môi trường xác minh trên cả 5 phiên", "Google Colab CPU-only (True), 2 vCPU, 12.975,52 MiB RAM"),
    ("Kiểm tra gối giờ và trùng tệp thống kê", f"sessions_overlap = {tot['sessions_overlap']} | duplicate_skipped = {tot['duplicate_run_files_skipped']}"),
    ("Tổng số bài báo phân tích mới từ đầu (reused = 0)", f"{fmt_int(tot['unique_publications'])} bài báo ({fmt_int(scope['versions'])} phiên bản)"),
    ("Tổng thời gian thực thi và kiểm chuẩn (5 phiên)", f"{fmt_num(tot['total_profiler_sec'], 2)} giây (5 giờ 11 phút 32 giây)"),
    ("Thời gian trung bình mỗi bài báo", f"{fmt_num(tot['avg_sec_per_unique_publication'], 4)} giây/bài (0,4504 giây/phiên bản)"),
    ("Phân vị thời gian xử lý một phiên bản có TeX", f"p50 = {fmt_num(ep['p50'], 3)}s | p90 = {fmt_num(ep['p90'], 3)}s | p99 = {fmt_num(ep['p99'], 3)}s | max = {fmt_num(ep['max'], 2)}s"),
    ("Đỉnh bộ nhớ RAM lớn nhất quan sát được (Peak RSS)", f"{fmt_num(tot['max_session_peak_rss_mib'], 2)} MiB (dưới 6,6% trần RAM máy ảo Colab)"),
    ("Bộ nhớ RAM trung bình gia quyền theo thời gian", f"{fmt_num(tot['session_time_weighted_avg_rss_mib'], 2)} MiB"),
    ("Đỉnh đĩa không gian làm việc tạm thời (kèm SQLite)", f"{fmt_num(tot['max_session_peak_disk_bytes']/1024**3, 2)} GiB ({fmt_int(tot['max_session_peak_disk_bytes'])} byte)"),
    ("Dung lượng thư mục parsed/ hợp nhất toàn nhóm", f"{fmt_num(tot['final_output_bytes']/1024**3, 2)} GiB ({fmt_int(tot['final_output_bytes'])} byte)"),
], columns=["Chỉ tiêu hiệu năng Milestone 2", "Giá trị xác minh"])
display(style_table(df_b8, caption="Bảng 8.1. Tổng hợp hiệu năng Milestone 2 trên Google Colab CPU-only (25.000 bài báo)."))

df_sess = pd.DataFrame(bench["sessions"])
view_s2 = pd.DataFrame({
    "Phiên": df_sess["session"],
    "Số bài": df_sess["processed_attempts"],
    "Thời gian (phút)": df_sess["profiler_sec"] / 60.0,
    "Giây/bài": df_sess["avg_sec_per_pub"],
    "Mean RAM (MiB)": df_sess["avg_rss_mib"],
    "Sampled Peak (MiB)": df_sess["peak_rss_mib"],
    "OS Peak (MiB)": df_sess["os_reported_peak_rss_mib"],
    "Đỉnh đĩa (GiB)": df_sess["peak_disk_bytes"] / (1024**3),
    "Parquet (GiB)": df_sess["final_output_bytes"] / (1024**3),
})
display(style_table(
    view_s2,
    caption="Bảng 8.2. Thông số tài nguyên từng phiên Colab CPU-only của 5 thành viên (0 bài tái sử dụng).",
    int_cols=["Số bài"],
))

ph_m2 = pd.DataFrame([
    {
        "Pha xử lý nội bộ M2": str(k),
        "Số lượt gọi": int(v["calls"]),
        "Tổng thời gian (giây)": float(v["elapsed_sec"]),
        "Tỷ trọng (%)": 100.0 * float(v["elapsed_sec"]) / bench["totals"]["total_profiler_sec"],
    }
    for k, v in bench["phases_aggregated"].items()
]).sort_values("Tổng thời gian (giây)", ascending=False)
display(style_table(
    ph_m2,
    caption="Bảng 8.3. Phân rã thời gian các pha xử lý nội bộ M2 cộng dồn qua 5 phiên.",
    int_cols=["Số lượt gọi"],
    pct_cols=["Tỷ trọng (%)"],
))

df_rec = pd.DataFrame(bench["storage_reconciliation"]).rename(columns={
    "tier": "Bậc lưu trữ và đóng gói",
    "bytes": "Dung lượng chính xác (Bytes)",
    "gib": "Quy đổi (GiB)",
    "note": "Giải thích kỹ thuật",
})
display(style_table(
    df_rec,
    caption="Bảng 8.4. Đối chiếu dung lượng lưu trữ giữa các bậc đầu ra từ M2 sang bộ dữ liệu 6 bảng trên Hugging Face.",
    int_cols=["Dung lượng chính xác (Bytes)"],
))

In [ ]:
df_s = pd.DataFrame(bench["sessions"])
fig, axes = plt.subplots(1, 3, figsize=(11.6, 3.7))

xb = np.arange(len(df_s))
m_lbls = [str(v) for v in df_s["member"]]
axes[0].bar(xb, df_s["avg_sec_per_pub"].to_numpy(dtype=float), color=BOOK_PALETTE["navy"], width=0.52)
axes[0].axhline(bench["totals"]["avg_sec_per_unique_publication"], color=BOOK_PALETTE["slate"], ls="--", lw=1.2, label="TB 0,7477s/bài")
axes[0].set_xticks(xb)
axes[0].set_xticklabels(m_lbls)
axes[0].set_ylim(0, 0.96)
axes[0].grid(axis="y")
axes[0].legend(fontsize=7.8)
axes[0].set_ylabel("Giây / bài báo")
axes[0].set_title("Hình 8.1a. Tốc độ trung bình mỗi bài")
for idx_i, v_i in enumerate(df_s["avg_sec_per_pub"]):
    axes[0].text(idx_i, float(v_i) + 0.02, fmt_num(v_i, 3), ha="center", fontsize=7.6)

wb = 0.25
axes[1].bar(xb - wb, df_s["avg_rss_mib"].to_numpy(dtype=float), wb, color="#94A3B8", label="Mean RSS")
axes[1].bar(xb, df_s["peak_rss_mib"].to_numpy(dtype=float), wb, color="#3B6E8C", label="Sampled Peak")
axes[1].bar(xb + wb, df_s["os_reported_peak_rss_mib"].to_numpy(dtype=float), wb, color="#1B365D", label="OS Peak")
axes[1].set_xticks(xb)
axes[1].set_xticklabels(m_lbls)
axes[1].set_ylim(0, 1050)
axes[1].grid(axis="y")
axes[1].legend(fontsize=7.5, loc="upper right")
axes[1].set_ylabel("RAM (MiB)")
axes[1].set_title("Hình 8.1b. Bộ nhớ RAM (< 6,6% trần Colab)")

axes[2].bar(xb - wb, (df_s["peak_disk_bytes"] / 1024**3).to_numpy(dtype=float), wb, color="#94A3B8", label="Đỉnh tạm thời")
axes[2].bar(xb, (df_s["final_disk_bytes"] / 1024**3).to_numpy(dtype=float), wb, color="#3B6E8C", label="Cuối phiên")
axes[2].bar(xb + wb, (df_s["final_output_bytes"] / 1024**3).to_numpy(dtype=float), wb, color="#1B365D", label="Parquet thuần")
axes[2].set_xticks(xb)
axes[2].set_xticklabels(m_lbls)
axes[2].set_ylim(0, 9.6)
axes[2].grid(axis="y")
axes[2].legend(fontsize=7.5, loc="upper right")
axes[2].set_ylabel("Dung lượng đĩa (GiB)")
axes[2].set_title("Hình 8.1c. Dung lượng đĩa mỗi phiên")
plt.tight_layout()
plt.show()

In [ ]:
profiles = bench.get("resource_profiles") or {}
avail = {k: v for k, v in profiles.items() if v and v.get("rows")}
if avail:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.0, 3.9))
    for label, prof in avail.items():
        f = pd.DataFrame(prof["rows"])
        t_min = (f["window_end_sec"] / 60.0).to_numpy(dtype=float)
        m_key = label.split()[-1].strip("()")
        col, ls = MEMBER_STYLES.get(m_key, ("#1B365D", "-"))
        ax1.plot(t_min, f["rss_sample_mean_mib"].to_numpy(dtype=float), label=label, color=col, ls=ls, lw=1.4)
        disk_rows = f.dropna(subset=["disk_peak_bytes"])
        if not disk_rows.empty:
            ax2.plot((disk_rows["window_end_sec"] / 60.0).to_numpy(dtype=float),
                     (disk_rows["disk_peak_bytes"] / 1024**3).to_numpy(dtype=float),
                     label=label, color=col, ls=ls, lw=1.4)
    ax1.set_xlabel("Thời gian chạy (phút)")
    ax1.set_ylabel("Mean RSS cửa sổ 60 giây (MiB)")
    ax1.set_title("Hình 8.2a. Chuỗi thời gian RAM trung bình (4 phiên có nhật ký 60s)")
    ax1.grid(True)
    ax1.legend(fontsize=7.8)

    ax2.set_xlabel("Thời gian chạy (phút)")
    ax2.set_ylabel("Dung lượng không gian làm việc (GiB)")
    ax2.set_title("Hình 8.2b. Biến thiên đĩa khi tạo chỉ mục SQLite cuối phiên")
    ax2.grid(True)
    ax2.legend(fontsize=7.8)
    plt.tight_layout()
    plt.show()

## Tài liệu và mã nguồn đối chiếu

- [Đặc tả chính thức Milestone 2 - Hierarchical Parsing and Standardization](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) và [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`src/parser/main.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/main.py).
- Mã nguồn triển khai: [`src/crawler/profiler.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/profiler.py).
- Mã nguồn triển khai: [`src/crawler/benchmark_sessions.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/benchmark_sessions.py).
- Mã nguồn triển khai: [`tools/m2_colab_runner.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/m2_colab_runner.py).